In [1]:
import pandas as pd
df = pd.read_pickle(r"c:\Users\igome\OneDrive - Universidad Politécnica de Madrid\Documents\Proyectos\Fermin\XFLR5data\xflr5_pressures_combined.pkl")


In [2]:
df.head()

,source_file,airfoil_name,reynolds_million,reynolds,mach,ncrit,flap_deg,alpha_deg,cd,cl,cm,xtr1,xtr2,tehmom,cpmn,point_index,cp_intrados,cp_extrados,points_in_block,x_percent
0,Re_0.3_0.csv,NACA 2412 - 0,0.3,300000.0,0.0,9.0,0.0,-15.0,0.171,-0.556,-0.002,1.0,0.036,-0.0035,-2.0372,0,0.4425,-0.3253,101,0.0
1,Re_0.3_0.csv,NACA 2412 - 0,0.3,300000.0,0.0,9.0,0.0,-15.0,0.171,-0.556,-0.002,1.0,0.036,-0.0035,-2.0372,1,0.3005,-0.3086,101,1.0
2,Re_0.3_0.csv,NACA 2412 - 0,0.3,300000.0,0.0,9.0,0.0,-15.0,0.171,-0.556,-0.002,1.0,0.036,-0.0035,-2.0372,2,0.2534,-0.3029,101,2.0
3,Re_0.3_0.csv,NACA 2412 - 0,0.3,300000.0,0.0,9.0,0.0,-15.0,0.171,-0.556,-0.002,1.0,0.036,-0.0035,-2.0372,3,0.2210,-0.2950,101,3.0
4,Re_0.3_0.csv,NACA 2412 - 0,0.3,300000.0,0.0,9.0,0.0,-15.0,0.171,-0.556,-0.002,1.0,0.036,-0.0035,-2.0372,4,0.1994,-0.2872,101,4.0


## Entrenamiento de una MLP para predecir Cp

La red usa como entrada `reynolds`, `alpha_deg`, `flap_deg` y `x_percent`, y como salida `cp_intrados` y `cp_extrados`.

El conjunto de test se separa por casos aerodinámicos completos `(reynolds, alpha_deg, flap_deg)` para evitar fuga de información entre entrenamiento y evaluación.


In [3]:
from pathlib import Path

from joblib import dump
from sklearn.metrics import mean_absolute_error, root_mean_squared_error, r2_score
from sklearn.model_selection import GroupShuffleSplit
from sklearn.neural_network import MLPRegressor
from sklearn.preprocessing import StandardScaler

feature_cols = ["reynolds", "alpha_deg", "flap_deg", "x_percent"]
target_cols = ["cp_intrados", "cp_extrados"]

model_df = df[feature_cols + target_cols].dropna().copy()
groups = pd.util.hash_pandas_object(
    model_df[["reynolds", "alpha_deg", "flap_deg"]],
    index=False,
)

splitter = GroupShuffleSplit(n_splits=1, test_size=0.2, random_state=42)
train_idx, test_idx = next(
    splitter.split(model_df[feature_cols], model_df[target_cols], groups=groups)
)

train_df = model_df.iloc[train_idx].reset_index(drop=True)
test_df = model_df.iloc[test_idx].reset_index(drop=True)

X_train = train_df[feature_cols]
X_test = test_df[feature_cols]
y_train = train_df[target_cols]
y_test = test_df[target_cols]

print(f"Filas train: {len(train_df):,}")
print(f"Filas test:  {len(test_df):,}")
print(
    "Casos train:",
    train_df[["reynolds", "alpha_deg", "flap_deg"]].drop_duplicates().shape[0],
)
print(
    "Casos test: ",
    test_df[["reynolds", "alpha_deg", "flap_deg"]].drop_duplicates().shape[0],
)


Filas train: 296,385
Filas test:  74,131
Casos train: 2883
Casos test:  721


In [4]:
x_scaler = StandardScaler()
y_scaler = StandardScaler()

X_train_scaled = x_scaler.fit_transform(X_train)
X_test_scaled = x_scaler.transform(X_test)

y_train_scaled = y_scaler.fit_transform(y_train)
y_test_scaled = y_scaler.transform(y_test)


In [5]:
mlp = MLPRegressor(
    hidden_layer_sizes=(128, 128, 64),
    activation="relu",
    solver="adam",
    alpha=1e-4,
    batch_size=2048,
    learning_rate_init=1e-3,
    max_iter=200,
    early_stopping=True,
    validation_fraction=0.1,
    n_iter_no_change=20,
    random_state=42,
    verbose=True,
)

mlp.fit(X_train_scaled, y_train_scaled)

print(f"Iteraciones ejecutadas: {mlp.n_iter_}")
print(f"Loss final (escala normalizada): {mlp.loss_:.6f}")


Iteration 1, loss = 0.24507564
Validation score: 0.864326
Iteration 2, loss = 0.03342335
Validation score: 0.961096
Iteration 3, loss = 0.01439873
Validation score: 0.976019
Iteration 4, loss = 0.00980238
Validation score: 0.981400
Iteration 5, loss = 0.00780147
Validation score: 0.984445
Iteration 6, loss = 0.00658564
Validation score: 0.986670
Iteration 7, loss = 0.00581948
Validation score: 0.988435
Iteration 8, loss = 0.00532951
Validation score: 0.988711
Iteration 9, loss = 0.00488412
Validation score: 0.989996
Iteration 10, loss = 0.00451063
Validation score: 0.989453
Iteration 11, loss = 0.00427206
Validation score: 0.990903
Iteration 12, loss = 0.00391785
Validation score: 0.992234
Iteration 13, loss = 0.00363059
Validation score: 0.992277
Iteration 14, loss = 0.00351148
Validation score: 0.993271
Iteration 15, loss = 0.00321846
Validation score: 0.993342
Iteration 16, loss = 0.00309909
Validation score: 0.993871
Iteration 17, loss = 0.00293843
Validation score: 0.994300
Iterat

In [6]:
y_pred_scaled = mlp.predict(X_test_scaled)
y_pred = pd.DataFrame(
    y_scaler.inverse_transform(y_pred_scaled),
    columns=target_cols,
    index=y_test.index,
)

metricas = pd.DataFrame(
    {
        "MAE": [mean_absolute_error(y_test[col], y_pred[col]) for col in target_cols],
        "RMSE": [
            root_mean_squared_error(y_test[col], y_pred[col]) for col in target_cols
        ],
        "R2": [r2_score(y_test[col], y_pred[col]) for col in target_cols],
    },
    index=target_cols,
)

metricas


,MAE,RMSE,R2
cp_intrados,0.055610,0.082352,0.999268
cp_extrados,0.045681,0.085095,0.996110


In [7]:
model_path = Path("mlp_cp_model.joblib")
dump(
    {
        "model": mlp,
        "x_scaler": x_scaler,
        "y_scaler": y_scaler,
        "feature_cols": feature_cols,
        "target_cols": target_cols,
    },
    model_path,
)


def predict_cp(reynolds, alpha_deg, flap_deg, x_percent):
    X_new = pd.DataFrame(
        [
            {
                "reynolds": reynolds,
                "alpha_deg": alpha_deg,
                "flap_deg": flap_deg,
                "x_percent": x_percent,
            }
        ]
    )[feature_cols]
    y_new_scaled = mlp.predict(x_scaler.transform(X_new))
    y_new = y_scaler.inverse_transform(y_new_scaled)
    return pd.DataFrame(y_new, columns=target_cols)


print(f"Modelo guardado en: {model_path.resolve()}")
predict_cp(reynolds=300000, alpha_deg=4.0, flap_deg=9.0, x_percent=25.0)


Modelo guardado en: C:\Users\igome\OneDrive - Universidad Politécnica de Madrid\Documents\Proyectos\Fermin\XFLR5data\mlp_cp_model.joblib


,cp_intrados,cp_extrados
0,-0.986326,-0.90101


## Visualizacion de curvas de presion con la MLP entrenada

Prediccion de `cp_intrados` y `cp_extrados` para el caso `Re=0.5e6`, `flap=0`, `alpha=5`, comparando la salida de la red con los datos reales disponibles en el dataset.


In [ ]:
import joblib
import matplotlib.pyplot as plt
import numpy as np

bundle = joblib.load("mlp_cp_model.joblib")
mlp_loaded = bundle["model"]
x_scaler_loaded = bundle["x_scaler"]
y_scaler_loaded = bundle["y_scaler"]
feature_cols_loaded = bundle["feature_cols"]
target_cols_loaded = bundle["target_cols"]

reynolds_plot = 0.5e6
alpha_plot = 5.0
flap_plot = 0.0
x_percent_plot = np.arange(0.0, 101.0, 1.0)

X_plot = pd.DataFrame(
    {
        "reynolds": reynolds_plot,
        "alpha_deg": alpha_plot,
        "flap_deg": flap_plot,
        "x_percent": x_percent_plot,
    }
)[feature_cols_loaded]

y_plot_pred = y_scaler_loaded.inverse_transform(
    mlp_loaded.predict(x_scaler_loaded.transform(X_plot))
)
pred_plot_df = pd.DataFrame(y_plot_pred, columns=target_cols_loaded)
pred_plot_df["x_percent"] = x_percent_plot

actual_plot_df = df[
    (df["reynolds"] == reynolds_plot)
    & (df["alpha_deg"] == alpha_plot)
    & (df["flap_deg"] == flap_plot)
][["x_percent", "cp_intrados", "cp_extrados"]].copy()

plt.figure(figsize=(10, 6))
plt.plot(
    pred_plot_df["x_percent"],
    pred_plot_df["cp_intrados"],
    label="Intrados MLP",
    color="#d1495b",
    linewidth=2.5,
)
plt.plot(
    pred_plot_df["x_percent"],
    pred_plot_df["cp_extrados"],
    label="Extrados MLP",
    color="#00798c",
    linewidth=2.5,
)

if not actual_plot_df.empty:
    plt.plot(
        actual_plot_df["x_percent"],
        actual_plot_df["cp_intrados"],
        "--",
        label="Intrados datos",
        color="#d1495b",
        alpha=0.7,
    )
    plt.plot(
        actual_plot_df["x_percent"],
        actual_plot_df["cp_extrados"],
        "--",
        label="Extrados datos",
        color="#00798c",
        alpha=0.7,
    )

plt.gca().invert_yaxis()
plt.xlabel("x/c (%)")
plt.ylabel("Cp")
plt.title("Curvas de presion: Re=0.5e6, flap=0 deg, alpha=5 deg")
plt.grid(True, alpha=0.3)
plt.legend()
plt.tight_layout()
plt.show()
